In [4]:
from pathlib import Path
import pandas as pd

daily_path = Path("../data/daily/RELIANCE.parquet")

df_daily = pd.read_parquet(daily_path)

print(df_daily.head())
print(df_daily.info())
print(df_daily.shape)


         date    open    high     low   close    volume
0  2020-06-01  705.40  733.20  703.45  724.60  38678282
1  2020-06-02  727.30  734.00  724.80  731.90  21452170
2  2020-06-03  736.35  743.50  730.80  734.75  24577568
3  2020-06-04  735.90  757.55  734.45  752.90  33119210
4  2020-06-05  760.20  771.15  750.00  753.85  32040366
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1511 entries, 0 to 1510
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    1511 non-null   object 
 1   open    1511 non-null   float64
 2   high    1511 non-null   float64
 3   low     1511 non-null   float64
 4   close   1511 non-null   float64
 5   volume  1511 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 71.0+ KB
None
(1511, 6)


In [27]:
minute_path = Path("../data/minute/RELIANCE.parquet")

df_minute = pd.read_parquet(minute_path)

print(df_minute.head())
# print(df_minute.info())
print(df_minute.shape)

            timestamp   open    high     low  close  volume
0 2020-06-01 09:15:00  705.4  706.05  703.45  704.7  772062
1 2020-06-01 09:16:00  704.7  705.60  703.80  703.9  361144
2 2020-06-01 09:17:00  703.8  706.30  703.70  706.3  310700
3 2020-06-01 09:18:00  706.8  708.65  706.00  708.1  265712
4 2020-06-01 09:19:00  708.5  709.90  707.00  708.8  256512
(563457, 6)


In [8]:
print("Daily columns:", df_daily.columns.tolist())
print("Minute columns:", df_minute.columns.tolist())

print("Daily date range:", df_daily["date"].min(), "to", df_daily["date"].max())
print("Minute date range:", df_minute["timestamp"].min(), "to", df_minute["timestamp"].max())

Daily columns: ['date', 'open', 'high', 'low', 'close', 'volume']
Minute columns: ['timestamp', 'open', 'high', 'low', 'close', 'volume']
Daily date range: 2020-06-01 to 2026-06-30
Minute date range: 2020-06-01 09:15:00 to 2026-06-29 15:29:00


In [18]:
print("Daily missing values:")
print(df_daily.isna().sum())

print("Daily duplicate dates:", df_daily["date"].duplicated().sum())

print("Minute missing values:")
print(df_minute.isna().sum())

print("Minute duplicate timestamps:", df_minute["timestamp"].duplicated().sum())

print("Zero-volume minute bars:", (df_minute["volume"] == 0).sum())

Daily missing values:
date      0
open      0
high      0
low       0
close     0
volume    0
dtype: int64
Daily duplicate dates: 0
Minute missing values:
timestamp    0
open         0
high         0
low          0
close        0
volume       0
dtype: int64
Minute duplicate timestamps: 0
Zero-volume minute bars: 9


In [19]:
from pathlib import Path
import pandas as pd

daily_dir = Path("../data/daily")

daily_summaries = []

for file in sorted(daily_dir.glob("*.parquet")):
    df = pd.read_parquet(file)

    daily_summaries.append({
        "symbol": file.stem,
        "rows": len(df),
        "start_date": df["date"].min(),
        "end_date": df["date"].max(),
        "duplicate_dates": df["date"].duplicated().sum(),
        "missing_values": int(df.isna().sum().sum()),
        "non_positive_open": int((df["open"] <= 0).sum()),
        "non_positive_high": int((df["high"] <= 0).sum()),
        "non_positive_low": int((df["low"] <= 0).sum()),
        "non_positive_close": int((df["close"] <= 0).sum()),
        "negative_volume": int((df["volume"] < 0).sum()),
        "zero_volume_days": int((df["volume"] == 0).sum()),
    })

daily_summary = pd.DataFrame(daily_summaries)

daily_summary.head()

,symbol,rows,start_date,end_date,duplicate_dates,missing_values,non_positive_open,non_positive_high,non_positive_low,non_positive_close,negative_volume,zero_volume_days
0,360ONE,1511,2020-06-01,2026-06-30,0,0,0,0,0,0,0,0
1,ABB,1511,2020-06-01,2026-06-30,0,0,0,0,0,0,0,0
2,ABCAPITAL,1511,2020-06-01,2026-06-30,0,0,0,0,0,0,0,0
3,ADANIENSOL,1511,2020-06-01,2026-06-30,0,0,0,0,0,0,0,0
4,ADANIENT,1511,2020-06-01,2026-06-30,0,0,0,0,0,0,0,0


In [20]:
print("Number of symbols:", daily_summary["symbol"].nunique())
print("Row count range:", daily_summary["rows"].min(), "to", daily_summary["rows"].max())
print("Overall start date:", daily_summary["start_date"].min())
print("Overall end date:", daily_summary["end_date"].max())

print("\nSymbols with duplicate dates:")
display(daily_summary[daily_summary["duplicate_dates"] > 0])

print("\nSymbols with missing values:")
display(daily_summary[daily_summary["missing_values"] > 0])

print("\nSymbols with invalid prices:")
display(
    daily_summary[
        (daily_summary["non_positive_open"] > 0)
        | (daily_summary["non_positive_high"] > 0)
        | (daily_summary["non_positive_low"] > 0)
        | (daily_summary["non_positive_close"] > 0)
    ]
)

print("\nSymbols with negative volume:")
display(daily_summary[daily_summary["negative_volume"] > 0])

Number of symbols: 208
Row count range: 378 to 1511
Overall start date: 2020-06-01
Overall end date: 2026-06-30

Symbols with duplicate dates:


,symbol,rows,start_date,end_date,duplicate_dates,missing_values,non_positive_open,non_positive_high,non_positive_low,non_positive_close,negative_volume,zero_volume_days



Symbols with missing values:


,symbol,rows,start_date,end_date,duplicate_dates,missing_values,non_positive_open,non_positive_high,non_positive_low,non_positive_close,negative_volume,zero_volume_days



Symbols with invalid prices:


,symbol,rows,start_date,end_date,duplicate_dates,missing_values,non_positive_open,non_positive_high,non_positive_low,non_positive_close,negative_volume,zero_volume_days



Symbols with negative volume:


,symbol,rows,start_date,end_date,duplicate_dates,missing_values,non_positive_open,non_positive_high,non_positive_low,non_positive_close,negative_volume,zero_volume_days


In [21]:
minute_dir = Path("../data/minute")

minute_summaries = []

for file in sorted(minute_dir.glob("*.parquet")):
    df = pd.read_parquet(file)

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["date"] = df["timestamp"].dt.date

    bars_per_day = df.groupby("date").size()

    minute_summaries.append({
        "symbol": file.stem,
        "rows": len(df),
        "start_timestamp": df["timestamp"].min(),
        "end_timestamp": df["timestamp"].max(),
        "trading_days": df["date"].nunique(),
        "duplicate_timestamps": df["timestamp"].duplicated().sum(),
        "missing_values": int(df.isna().sum().sum()),
        "zero_volume_bars": int((df["volume"] == 0).sum()),
        "non_positive_prices": int(
            (
                (df["open"] <= 0)
                | (df["high"] <= 0)
                | (df["low"] <= 0)
                | (df["close"] <= 0)
            ).sum()
        ),
        "days_with_375_bars": int((bars_per_day == 375).sum()),
        "days_below_375_bars": int((bars_per_day < 375).sum()),
        "days_above_375_bars": int((bars_per_day > 375).sum()),
        "minimum_bars_in_day": int(bars_per_day.min()),
        "maximum_bars_in_day": int(bars_per_day.max()),
    })

minute_summary = pd.DataFrame(minute_summaries)

In [23]:
print("Number of minute symbols:", minute_summary["symbol"].nunique())
print("Row count range:", minute_summary["rows"].min(), "to", minute_summary["rows"].max())
print("Trading day range:", minute_summary["trading_days"].min(), "to", minute_summary["trading_days"].max())

print("\nSymbols with duplicate timestamps:")
display(minute_summary[minute_summary["duplicate_timestamps"] > 0])

print("\nSymbols with incomplete sessions:")
display(
    minute_summary[
        (minute_summary["days_below_375_bars"] > 0)
        | (minute_summary["days_above_375_bars"] > 0)
    ][
        [
            "symbol",
            "days_below_375_bars",
            "days_above_375_bars",
            "minimum_bars_in_day",
            "maximum_bars_in_day",
        ]
    ]
)

Number of minute symbols: 208
Row count range: 141032 to 563581
Trading day range: 377 to 1509

Symbols with duplicate timestamps:


,symbol,rows,start_timestamp,end_timestamp,trading_days,duplicate_timestamps,missing_values,zero_volume_bars,non_positive_prices,days_with_375_bars,days_below_375_bars,days_above_375_bars,minimum_bars_in_day,maximum_bars_in_day



Symbols with incomplete sessions:


,symbol,days_below_375_bars,days_above_375_bars,minimum_bars_in_day,maximum_bars_in_day
0,360ONE,535,0,59,375
1,ABB,62,0,60,375
2,ABCAPITAL,18,0,60,375
3,ADANIENSOL,75,0,60,375
4,ADANIENT,17,0,60,375
...,...,...,...,...,...
203,VOLTAS,18,0,60,375
204,WAAREEENER,5,0,60,375
205,WIPRO,17,0,60,375
206,YESBANK,17,0,60,375


In [24]:
daily_symbols = set(daily_summary["symbol"])
minute_symbols = set(minute_summary["symbol"])

print("Only in daily:", sorted(daily_symbols - minute_symbols))
print("Only in minute:", sorted(minute_symbols - daily_symbols))

Only in daily: []
Only in minute: []


In [25]:
daily_summary.to_csv("../outputs/daily_data_summary.csv", index=False)
minute_summary.to_csv("../outputs/minute_data_summary.csv", index=False)